<h2 style="color:#8E44AD; margin-bottom:6px;">
  02 – Tuned TF-IDF + One-vs-Rest Logistic Regression (RandomizedSearchCV)
</h2>

<p style="font-size:16px; margin-top:4px;">
  <strong>Overview:</strong>
  This notebook builds on the traditional TF-IDF + Logistic Regression baseline
  by applying hyperparameter tuning using <strong>RandomizedSearchCV</strong>
  on a pipeline that combines TF-IDF vectorization with a One-vs-Rest classifier.
</p>

<p style="font-size:16px;">
  <strong>Goal:</strong>
  Improve the Macro F1 score of the baseline model by searching over
  different TF-IDF configurations and regularization strengths for Logistic
  Regression, while keeping the model simple and interpretable.
</p>

<p style="font-size:16px;">
  Cross-validation is used to estimate performance during tuning, and a
  separate holdout validation split is used to verify the tuned model before
  training on the full dataset and generating the final Kaggle submission.
</p>


In [ ]:
#wandb login check to make sure everything goes smooth

from kaggle_secrets import UserSecretsClient
import wandb

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=api_key)

print("W&B login successful!")

In [3]:
# 02 – Tuned TF-IDF + One-vs-Rest Logistic Regression (RandomizedSearchCV)

import pandas as pd
import wandb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from scipy.stats import uniform

# --------------------------------------------------------
# 1. W&B INIT
# --------------------------------------------------------
wandb.init(
    project="24ds2000048-t32025",
    entity="sharmila-m",
    name="02_tfidf_logistic_regression_ovr_tuned",
    config={
        "model": "OneVsRest Logistic Regression",
        "method": "RandomizedSearchCV",
        "n_iter": 10,
        "cv": 3,
        "test_size": 0.2,
        "random_state": 42
    }
)

config = wandb.config

# --------------------------------------------------------
# 2. Load Data
# --------------------------------------------------------
train = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/train.csv")
test = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")

X = train["text"]
y = train[["anger", "fear", "joy", "sadness", "surprise"]]

# --------------------------------------------------------
# 3. Train/Validation Split
# --------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=config.test_size,
    random_state=config.random_state
)

# --------------------------------------------------------
# 4. Pipeline (TF-IDF + OVR Logistic Regression)
# --------------------------------------------------------
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", OneVsRestClassifier(
        LogisticRegression(solver="liblinear")
    ))
])

# --------------------------------------------------------
# 5. Hyperparameter Search Space
# --------------------------------------------------------
param_distributions = {
    "tfidf__max_features": [3000, 5000, 8000, 12000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__estimator__C": uniform(0.01, 10),
}

# --------------------------------------------------------
# 6. RandomizedSearchCV (Tuning)
# --------------------------------------------------------
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=config.n_iter,
    cv=config.cv,
    scoring="f1_macro",
    random_state=config.random_state,
    n_jobs=-1,
    verbose=2,
)

search.fit(X_train, y_train)

# --------------------------------------------------------
# 7. Log best CV results to W&B
# --------------------------------------------------------
wandb.log({
    "cv_best_f1_macro": search.best_score_,
})

# log best params as separate keys
for param_name, param_value in search.best_params_.items():
    wandb.log({f"cv_best_param_{param_name}": param_value})

print("Best Cross-Validation F1 (macro):", search.best_score_)
print("Best Params:", search.best_params_)

# --------------------------------------------------------
# 8. Holdout validation (using best estimator)
# --------------------------------------------------------
val_preds = search.best_estimator_.predict(X_val)

val_f1_macro = f1_score(y_val, val_preds, average="macro")
val_accuracy = accuracy_score(y_val, val_preds)

print("Holdout Validation Macro F1:", val_f1_macro)
print("Holdout Validation Accuracy:", val_accuracy)

wandb.log({
    "val_f1_macro": val_f1_macro,
    "val_accuracy": val_accuracy
})

# --------------------------------------------------------
# 9. Train best model on FULL DATA
# --------------------------------------------------------
best_model = search.best_estimator_
best_model.fit(X, y)

wandb.finish()

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Cross-Validation F1 (macro): 0.5888681049731744
Best Params: {'clf__estimator__C': 6.021150117432088, 'tfidf__max_features': 12000, 'tfidf__ngram_range': (1, 1)}
Holdout Validation Macro F1: 0.6729019477448122
Holdout Validation Accuracy: 0.5161054172767203


cv_best_f1_macro,▁
cv_best_param_clf__estimator__C,▁
cv_best_param_tfidf__max_features,▁
val_accuracy,▁
val_f1_macro,▁
cv_best_f1_macro,0.58887
cv_best_param_clf__estimator__C,6.02115
cv_best_param_tfidf__max_features,12000
val_accuracy,0.51611
val_f1_macro,0.6729


[CV] END clf__estimator__C=3.7554011884736247, tfidf__max_features=3000, tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END clf__estimator__C=7.3299394181140505, tfidf__max_features=3000, tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END clf__estimator__C=0.5908361216819946, tfidf__max_features=12000, tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END clf__estimator__C=6.021150117432088, tfidf__max_features=12000, tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END clf__estimator__C=0.21584494295802448, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.3s
[CV] END clf__estimator__C=8.334426408004218, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.6s
[CV] END clf__estimator__C=4.329450186421157, tfidf__max_features=3000, tfidf__ngram_range=(1, 1); total time=   0.3s
[CV] END clf__estimator__C=3.7554011884736247, tfidf__max_features=3000, tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END clf__estimator__C=1.5701864044243652, t

In [ ]:
# --------------------------------------------------------
# 10. Predict test set
# --------------------------------------------------------
test_preds = best_model.predict(test["text"])

# --------------------------------------------------------
# 11. Create submission
# --------------------------------------------------------
submission = pd.concat(
    [
        test["id"],
        pd.DataFrame(
            test_preds,
            columns=["anger", "fear", "joy", "sadness", "surprise"]
        )
    ],
    axis=1
)

submission_filename = "submission.csv"
submission.to_csv(submission_filename, index=False)
print(f"Saved {submission_filename}")

<hr/>

<h3 style="color:#8E44AD;">
  Results & Runtime Summary
</h3>

<p style="font-size:16px;">
  <strong>Validation Metrics (Internal):</strong><br/>
  Macro F1 Score: <strong>0.673</strong><br/>
  Accuracy: <strong>0.516</strong>
</p>

<p style="font-size:16px;">
  <strong>Kaggle Public Leaderboard Score:</strong><br/>
  Macro F1 Score: <strong>0.696</strong>
</p>

<p style="font-size:16px;">
  <strong>Runtime Environment:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li><strong>Platform:</strong> Kaggle Notebook</li>
  <li><strong>Hardware:</strong> CPU</li>
  <li><strong>Estimated Runtime:</strong> 48s</li>
  <li><strong>Libraries:</strong> scikit-learn, pandas</li>
</ul>


<p style="font-size:16px;">
  <strong>Observations:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li>Hyperparameter tuning produced a moderate improvement over the baseline model.</li>
  <li>Performance gains are limited due to the linear nature of TF-IDF-based models.</li>
  <li>The results motivated the transition to more expressive neural and transformer models.</li>
</ul>
